# قصه معنوی — Pure AI (بدون Mixkit)

1. Runtime → Change runtime type → **T4 GPU** → Save
2. Runtime → **Run all**
3. در پایان `final_story.mp4` را دانلود کن


In [ ]:
!nvidia-smi -L || echo 'GPU not visible — still OK, images use Pollinations'
!pip -q install edge-tts
!apt-q update >/dev/null 2>&1 && apt-q install -y ffmpeg fonts-noto-core >/dev/null 2>&1 || true
print('ready')

In [ ]:
import asyncio, subprocess, time, urllib.parse, urllib.request
from pathlib import Path

BEATS = [
    {
        'line': 'تو کوهستان، پسری فکر می‌کرد خدا خیلی دور است.',
        'prompt': '2D children storybook ONE young boy age 10 short brown hair standing alone alpine village looking far wooden houses green mountains soft daylight flat colors vertical 9:16 NO second person NO text NOT photo',
    },
    {
        'line': 'هر شب به آسمان پر ستاره نگاه می‌کرد.',
        'prompt': '2D children storybook SAME boy lying on grass looking UP huge starry night sky wonder mountain silhouette vertical 9:16 NO second person NO text',
    },
    {
        'line': 'می‌گفت کاش می‌تونستم به خدا برسم.',
        'prompt': '2D children storybook close-up SAME boy face looking at stars longing hope soft starlight vertical 9:16 NO second person',
    },
    {
        'line': 'یه روز طوفان اومد و یه بره از گله گم شد.',
        'prompt': '2D children storybook ONE small white lamb alone near rocks soft rain flock far away vertical 9:16 NO boy NO hugging NO text',
    },
    {
        'line': 'پسر رفت زیر باران دنبال بره.',
        'prompt': '2D children storybook SAME boy walking alone rain wet mountain path determined wet hair vertical 9:16 NO second person NO text',
    },
    {
        'line': 'بره را پیدا کرد و محکم بغلش کرد.',
        'prompt': '2D children storybook CRITICAL young boy kneeling hugging small white LAMB wool visible warm relief golden light vertical 9:16 FORBIDDEN two boys human baby missing lamb',
    },
    {
        'line': 'اون لحظه فهمید خدا در مهربانی کنار اوست.',
        'prompt': '2D children storybook SAME boy and white lamb under soft rainbow light green slope peaceful vertical 9:16 NO text',
    },
    {
        'line': 'نزدیک‌ترین راه به آسمان، یک قدم مهربانی است.',
        'prompt': '2D children storybook SAME boy walking home with lamb alpine village warm sunset orange pink sky vertical 9:16 NO text',
    },
]

STYLE = ' Same boy every scene short brown hair earth tunic age 10 pure 2D storybook soft alpine colors.'
OUT = Path('story_out'); OUT.mkdir(exist_ok=True)
(OUT/'images').mkdir(exist_ok=True)
(OUT/'audio').mkdir(exist_ok=True)

def run(cmd, t=180):
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=t)
    if r.returncode: raise RuntimeError((r.stderr or r.stdout)[-1200:])

def dur(p):
    r = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(p)], capture_output=True, text=True)
    return float(r.stdout.strip() or '0')

async def tts(text, path):
    import edge_tts
    await edge_tts.Communicate(text, 'fa-IR-FaridNeural', rate='-15%', pitch='-2Hz', volume='+50%').save(str(path))

def fetch(prompt, dest, seed):
    q = urllib.parse.quote(prompt + STYLE)
    for a in range(5):
        s = 8000 + seed*41 + a*19
        url = f'https://image.pollinations.ai/prompt/{q}?width=768&height=1344&nologo=true&seed={s}&model=flux'
        try:
            print(f'img {seed+1} try {a+1}')
            req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=100) as r, open(dest,'wb') as f:
                while True:
                    c = r.read(65536)
                    if not c: break
                    f.write(c)
            if dest.stat().st_size > 40000: return True
        except Exception as e:
            print(type(e).__name__); time.sleep(12)
            dest.unlink(missing_ok=True)
    return False

def ken(img, seconds, out, seed):
    frames = max(40, int(seconds*24))
    z = 'min(1.18,1+0.00035*on)' if seed%2==0 else "if(eq(on,1),1.16,max(1.0,1.16-0.00035*on))"
    run(['ffmpeg','-y','-loop','1','-i',str(img),'-t',f'{seconds:.3f}',
         '-vf', f"scale=960:1700:force_original_aspect_ratio=increase,zoompan=z='{z}':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={frames}:s=720x1280:fps=24",
         '-an','-c:v','libx264','-preset','fast','-crf','14','-pix_fmt','yuv420p', str(out)])

segs=[]; t=0.0; caps=[]
for i,b in enumerate(BEATS):
    raw=OUT/'audio'/f'r{i}.mp3'; ap=OUT/'audio'/f'a{i}.mp3'
    asyncio.run(tts(b['line'], raw))
    run(['ffmpeg','-y','-i',str(raw),'-af','loudnorm=I=-12:TP=-1.0:LRA=8','-ar','48000','-ac','1',str(ap)])
    d=max(1.6, dur(ap)); vd=d+0.3
    img=OUT/'images'/f's{i}.jpg'
    assert fetch(b['prompt'], img, i), f'img fail {i}'
    vo=OUT/f'v{i}.mp4'; ken(img, vd, vo, i)
    pad=OUT/'audio'/f'p{i}.mp3'
    run(['ffmpeg','-y','-i',str(ap),'-af','apad=pad_dur=0.3','-t',f'{vd:.3f}',str(pad)])
    seg=OUT/f'seg{i}.mp4'
    run(['ffmpeg','-y','-i',str(vo),'-i',str(pad),'-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-b:a','192k','-shortest',str(seg)])
    segs.append(seg); caps.append((t,t+vd,b['line'])); t+=vd
    print(f'ok {i+1}/{len(BEATS)}'); time.sleep(5)

clist=OUT/'c.txt'; clist.write_text('\n'.join(f"file '{p.resolve()}'" for p in segs))
asm=OUT/'asm.mp4'
run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(clist),'-c:v','libx264','-preset','fast','-crf','14','-c:a','aac','-b:a','192k','-pix_fmt','yuv420p',str(asm)])

# captions
def ts(x):
    h,r=divmod(max(0.0,x),3600); m,s=divmod(r,60); return f'{int(h)}:{int(m):02d}:{s:05.2f}'
ass=OUT/'c.ass'
hdr='''[Script Info]\nScriptType: v4.00+\nPlayResX: 720\nPlayResY: 1280\n\n[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\nStyle: Body,Noto Sans Arabic,52,&H00FFFFFF,&H000000FF,&H00000000,&HCC000000,0,0,0,0,100,100,0,0,1,6,3,2,22,22,165,1\n\n[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n'''
ev=[]
for a,b,line in caps:
    ev.append(f'Dialogue: 0,{ts(a+0.05)},{ts(max(a+0.5,b-0.05))},Body,,0,0,0,,{line}')
ass.write_text(hdr+'\n'.join(ev)+'\n', encoding='utf-8')
final=Path('final_story.mp4')
run(['ffmpeg','-y','-i',str(asm),'-vf',f'ass={ass.resolve().as_posix()}','-c:v','libx264','-preset','fast','-crf','13','-pix_fmt','yuv420p','-c:a','copy','-movflags','+faststart',str(final)])
print('DONE', final.resolve(), final.stat().st_size)
from google.colab import files
files.download('final_story.mp4')